#### Day 4 - Final Preprocessing Pipeline

Objective:

1. Finalize feature selection
2. Build reusable preprocessing workflow
3. Create train-test split
4. Generate model-ready datasets
5. Save preprocessing artifacts for future use

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

import joblib

In [33]:
DATA_PATH = "D:/Projects/fraud_detection_platform/data/raw/fraud_data.csv"

df = pd.read_csv(
    DATA_PATH,
    dtype={
        "step": "int32",
        "amount": "float32",
        "isFraud": "int8",
        "isFlaggedFraud": "int8"
    }
)

print(df.shape)
print(df.columns)

(6362620, 11)
Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')


#### Feature Selection Decisions
##### Dropped columns is nameOrig ,nameDest
##### Retained columns is type, amount, balance features, engineered features
##### isFlaggedFraud retained for baseline experiment.

In [25]:
df=df.drop(columns=["nameOrig","nameDest"],errors="ignore")
print(df.columns.tolist())

['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


##### Creating Engineered Features

In [29]:
#Balance change in sender bank account
df['balance_change_orig']=df['oldbalanceOrg']-df['newbalanceOrig']
#Balance change in reciever bank account
df['balance_change_dest']=df['oldbalanceDest']-df['newbalanceDest']
df['amount_to_balance_ratio']=(df['amount']/(df['oldbalanceOrg']+1))

In [ ]:
#The transaction amount feature was highly right-skewed.
# I applied a logarithmic transformation using np.log1p() to reduce skewness, minimize the influence of extreme values,
# and create a more model-friendly feature for training.EG:- 10000 value becomes 10.11 like this huge values are compressed
df['log_amount']=np.log1p(df['amount'])
# Hour_of_the_day feature is helpful to know when the transaction is occur wetheer day time or night
df['hour_of_day']=df['step']%24
# Day Feature is helpful ti know on which day of the month  transaction is occur.it can capture temporial patterns
df['day']=df['step']//24

##### Creating Feature And Target Matrix

In [36]:
X=df.drop("isFraud",axis=1)
Y=df["isFraud"]
print("Shape Of feature Matrix:- ",X.shape)
print("Shape Of target Matrix:- ",Y.shape)

Shape Of feature Matrix:-  (6362620, 12)
Shape Of target Matrix:-  (6362620,)


In [37]:
TARGET_COLUMN = "isFraud"
CATEGORICAL_FEATURES = ["type"]
NUMERICAL_FEATURES = [col
    for col in X.columns
    if col not in CATEGORICAL_FEATURES
]

In [39]:
# For Verifying Data Leakage 
pd.crosstab(df['isFlaggedFraud'],df['isFraud'],normalize='index')*100

isFraud,0,1
isFlaggedFraud,,
0,99.871169,0.128831
1,0.000000,100.000000


##### No leakage in data because whenever isflaggedfraud is true then the transaction is label as fraud

#### Creating Preprocessor

In [ ]:
# I used a ColumnTransformer to apply One-Hot Encoding to categorical features while passing numerical features unchanged. 
# The handle_unknown='ignore' parameter ensures robustness when unseen categories appear during inference, 
# and remainder='passthrough' preserves all numerical features for model training.
preprocessor=ColumnTransformer(transformers=[("categorical",OneHotEncoder(handle_unknown="ignore"),CATEGORICAL_FEATURES)],
                               remainder='passthrough')

##### Splitting Of Data for Train and Tesst the Model

In [42]:
#Stratify parameter is used for split the data with same distribution of fraud transcation in both train and test data
X_train,X_test,y_train,y_test=train_test_split(X,Y,test_size=0.20,random_state=42,stratify=Y)
print("Shape of train data:- ",X_train.shape)
print("Shape of test data:- ",X_test.shape)

Shape of train data:-  (5090096, 12)
Shape of test data:-  (1272524, 12)


In [44]:
#Fit the processor on data
X_train_processed=preprocessor.fit_transform(X_train)
X_test_processed=preprocessor.transform(X_test)
print("X_train_processed:",X_train_processed.shape)

print("X_test_processed:",X_test_processed.shape)

X_train_processed: (5090096, 16)
X_test_processed: (1272524, 16)


In [46]:
#For saving preprocessor
joblib.dump(preprocessor,"D:/Projects/fraud_detection_platform/artifacts/preprocessor.pkl")
print("Preprocessor is saved sucessfully")

Preprocessor is saved sucessfully


In [48]:
#Saving Preprocessed Data
joblib.dump(X_train_processed,"D:/Projects/fraud_detection_platform/artifacts/X_train.pkl")

joblib.dump(X_test_processed,"D:/Projects/fraud_detection_platform/artifacts/X_test.pkl")

joblib.dump(y_train,"D:/Projects/fraud_detection_platform/artifacts/y_train.pkl")

joblib.dump(y_test,"D:/Projects/fraud_detection_platform/artifacts/y_test.pkl")
print("Preprocessed data saved sucessfully")

Preprocessed data saved sucessfully


In [50]:
print("DAY 4 COMPLETE\n")

print("--Features Finalized")
print("--Preprocessing Pipeline Created")
print("--Train-Test Split Completed")
print("--Artifacts Saved")

DAY 4 COMPLETE

--Features Finalized
--Preprocessing Pipeline Created
--Train-Test Split Completed
--Artifacts Saved
